# DDC - MATLAB vs Python validation

Use the same input and scientific parameters in MATLAB and Python. Compare deterministic quantities directly, then compare stochastic MCMC outcomes without assuming that the same numeric seed produces the same random stream in MATLAB and NumPy.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import hdf5storage
from ddc import load_dataset, palm_image_squared_error

## 1. Paths

`INPUT_MAT` must be the same raw pre-v7.3 input used by both implementations. `MATLAB_RESULT` and `PYTHON_RESULT` are the analyzed outputs to compare.

In [ ]:
INPUT_MAT = Path("input_DDC_pre_v73.mat")
MATLAB_RESULT = Path("matlab_result.mat")
PYTHON_RESULT = Path("python_result.mat")
IMAGE_INDEX = 0

for p in (INPUT_MAT, MATLAB_RESULT, PYTHON_RESULT):
    assert p.exists(), p

## 2. Load result MAT files

In [ ]:
def load_result(path):
    return hdf5storage.loadmat(str(path))

matlab = load_result(MATLAB_RESULT)
python = load_result(PYTHON_RESULT)
dataset = load_dataset(INPUT_MAT)

print("MATLAB keys:", sorted(k for k in matlab if not k.startswith("__"))[:25], "...")
print("Python keys:", sorted(k for k in python if not k.startswith("__"))[:25], "...")

## 3. Helpers for MATLAB cells and numeric arrays

In [ ]:
def cell_at(mapping, name, image_index=0):
    cells = np.asarray(mapping[name], dtype=object).ravel(order="F")
    return np.asarray(cells[image_index])

def numeric(mapping, name):
    return np.asarray(mapping[name])

def compare_numeric(name, rtol=1e-12, atol=1e-12):
    a = numeric(matlab, name)
    b = numeric(python, name)
    same_shape = a.shape == b.shape
    ok = same_shape and np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True)
    max_abs = np.nan if not same_shape else np.nanmax(np.abs(a.astype(float) - b.astype(float))) if a.size else 0.0
    print(f"{name:26s} shape_equal={same_shape} allclose={ok} max_abs={max_abs}")
    return ok

### Check that the spatial regions match

An image index identifies a cell, not an entire original image. Different splitting settings can put different regions at the same index. Check the saved raw inputs before comparing distributions, counts, or plots.


In [ ]:
def validate_input_regions(mapping, label):
    cells = np.asarray(mapping["LocalizationsFinal"], dtype=object).ravel(order="F")
    print(f"{label}: {len(cells)} input regions; reference: {dataset.n_images}")
    if len(cells) != dataset.n_images:
        raise ValueError(
            f"{label} has {len(cells)} regions, but INPUT_MAT has {dataset.n_images}. "
            "The files use different spatial splits. Run both implementations on "
            "the exact same split input before comparing cells by IMAGE_INDEX."
        )
    for index, (saved, original) in enumerate(zip(cells, dataset.localizations)):
        saved = np.asarray(saved)
        original = np.asarray(original)
        # Both pipelines may pad 2D coordinates with a zero z column.
        if original.ndim == 2 and original.shape[1] == 2:
            original = np.column_stack((original, np.zeros(len(original))))
        if saved.ndim == 2 and saved.shape[1] == 2:
            saved = np.column_stack((saved, np.zeros(len(saved))))
        if saved.shape != original.shape or not np.allclose(
            saved, original, rtol=0, atol=1e-10, equal_nan=True
        ):
            raise ValueError(
                f"{label} region {index} differs from INPUT_MAT (shape, coordinates, "
                "or row order). Use the same input and preprocessing for both runs."
            )

validate_input_regions(matlab, "MATLAB")
validate_input_regions(python, "Python")


## 4. Compare deterministic pre-MCMC quantities first

If these disagree materially, stop before interpreting stochastic MCMC results.

In [ ]:
for name in ["bins", "Distribution_for_Blink", "X_overall", "M_mat"]:
    if name in matlab and name in python:
        compare_numeric(name)
    else:
        print(name, "missing from one result file")

## 5. Extract selected-image results

In [ ]:
raw = dataset.localizations[IMAGE_INDEX]
truth = dataset.true_localizations[IMAGE_INDEX]

matlab_final = cell_at(matlab, "Final_Loc_Blinking_Corrected", IMAGE_INDEX)
python_final = cell_at(python, "Final_Loc_Blinking_Corrected", IMAGE_INDEX)
matlab_lik = cell_at(matlab, "Lik", IMAGE_INDEX).reshape(-1)
python_lik = cell_at(python, "Lik", IMAGE_INDEX).reshape(-1)

print("Raw:", len(raw))
print("Truth:", len(truth))
print("MATLAB final:", len(matlab_final))
print("Python final:", len(python_final))
print("MATLAB best likelihood:", np.nanmax(matlab_lik))
print("Python best likelihood:", np.nanmax(python_lik))

## 6. Compare counting error against truth

In [ ]:
if truth.size:
    n_true = len(truth)
    metrics = {
        "Raw": len(raw) - n_true,
        "MATLAB": len(matlab_final) - n_true,
        "Python": len(python_final) - n_true,
    }
    for name, err in metrics.items():
        print(f"{name:7s}: signed error={err:+d}, abs relative error={abs(err)/n_true:.4%}")

## 7. Visual comparison

A single stochastic run is descriptive, not a proof of MCMC equivalence. Use it to inspect gross reconstruction differences.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
for ax, data, title in [
    (axes[0], raw, f"Raw - n={len(raw)}"),
    (axes[1], truth, f"Truth - n={len(truth)}"),
    (axes[2], matlab_final, f"MATLAB - n={len(matlab_final)}"),
    (axes[3], python_final, f"Python - n={len(python_final)}"),
]:
    ax.scatter(data[:,0], data[:,1], s=2)
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
plt.tight_layout()

## 8. Repeated-run stochastic validation

For a serious MATLAB-vs-Python MCMC comparison, run many independent replicates in each language. Do not pair MATLAB seed `s` with NumPy seed `s` as if the trajectories should be identical. Compare distributions of final counts, truth-based scores, trajectory-size summaries and other scientific outputs.

In [ ]:
# Example table schema for repeated runs:
# columns = [
#     "language", "seed", "final_count", "count_error",
#     "best_likelihood", "best_score", "n_trajectories",
#     "mean_trajectory_size", "max_trajectory_size"
# ]
# Save one CSV row per run in MATLAB and Python, then compare distributions.